# DDPM／DDIM 裁切消融：整理版

推甄成果的實驗附件。原始執行紀錄保留在 [ddpm_experiments.ipynb](https://github.com/yjjj3/ddpm-from-scratch/blob/main/ddpm_experiments.ipynb)。本整理版未重新執行，不包含移植或偽造的輸出。

預設只重建已保存結果的圖表，不掛載 Drive、不訓練、不生成影像。完整重跑需自行啟用對應開關與 GPU；權重不包含於儲存庫。

研究觀察：两份模型各三個取樣 seeds 的共同設定中，原本裁切在 20 步最低，替代方式在 50 步最低。這不代表普遍最佳步數，也未證明誤差機制。

## 1. 設定與固定版本

先依序執行。預設開關均為 False；「全部執行」只會整理既有結果。啟用新訓練時請選擇新的 TRAIN_SEED；既有權重一律不覆寫。

In [ ]:
RUN_TRAINING = False
RUN_ABLATION = False
TRAIN_SEED = 2026
SAMPLING_SEED = 123
EVAL_STEPS = [20, 50, 200]
# 原研究新增模型；如需評估另一模型，明確修改此路徑。
CHECKPOINT = "/content/drive/MyDrive/ddpm_independent/train_seed_2026/latest.pt"
SOURCE_COMMIT = "816843df7cb75a7784b70c0e1c55b5bce2aad3c0"


In [ ]:
import os, sys, subprocess
from pathlib import Path
repo = Path("/content/ddpm_review_" + SOURCE_COMMIT[:12])
if not repo.exists():
    subprocess.run(["git", "clone", "https://github.com/yjjj3/ddpm-from-scratch.git", str(repo)], check=True)
actual = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip()
if actual != SOURCE_COMMIT:
    if subprocess.check_output(["git", "-C", str(repo), "status", "--porcelain"], text=True).strip():
        raise RuntimeError("工作目錄有修改，請保留修改並另用乾淨資料夾")
    subprocess.run(["git", "-C", str(repo), "checkout", "--detach", SOURCE_COMMIT], check=True)
os.chdir(repo)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib"], check=True)
print("Source commit:", SOURCE_COMMIT)


## 2. 重建已保存結果（不需 GPU）

各模型內以三個取樣 seeds 計算平均與樣本標準差，不將六次取樣當成六次獨立訓練。

In [ ]:
subprocess.run([sys.executable, "scripts/compare_checkpoints.py"], check=True)
from IPython.display import display, SVG, Markdown
display(SVG(filename="assets/checkpoint_comparison.svg"))
display(Markdown(Path("results/clipping_ablation/checkpoint_tables.md").read_text()))


## 3. 可選：GPU 與權重環境

只有啟用訓練或消融時才掛載 Drive。依賴未完整鎖版，歷史結果不保證逐位元重現。

In [ ]:
if RUN_TRAINING or RUN_ABLATION:
    import torch
    assert torch.cuda.is_available(), "請切換至 GPU 執行階段"
    from google.colab import drive
    drive.mount("/content/drive")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "clean-fid"], check=True)
    print(torch.__version__, torch.cuda.get_device_name(0))
else:
    print("已跳過 GPU 與 Drive 設定")


## 4. 可選：新模型訓練

沿用原始訓練程式與隨機種子設定。已有 latest.pt 時跳過，不自動續訓；原訓練實作未完整保存 RNG／GradScaler 狀態。

In [ ]:
def train_new_model(training_seed):
    import importlib
    import random
    import json
    import subprocess
    import numpy as np
    import torch
    from pathlib import Path
    
    import ddpm_mnist
    importlib.reload(ddpm_mnist)
    
    TRAIN_SEED = training_seed
    
    train_dir = Path(
        f"/content/drive/MyDrive/ddpm_independent/train_seed_{TRAIN_SEED}"
    )
    train_dir.mkdir(parents=True, exist_ok=True)
    
    # Existing checkpoints are never overwritten.
    if (train_dir / "latest.pt").exists():
        print("已存在權重，跳過訓練：", train_dir)
        return
    assert torch.cuda.is_available(), "需要 GPU"
    
    random.seed(TRAIN_SEED)
    np.random.seed(TRAIN_SEED)
    torch.manual_seed(TRAIN_SEED)
    torch.cuda.manual_seed_all(TRAIN_SEED)
    
    class ExperimentCFG(ddpm_mnist.CFG):
        ckpt_dir = str(train_dir)
        device = "cuda"
        total_steps = 30_000
    
    # 保存設定，方便追溯
    config = {
        key: getattr(ExperimentCFG, key)
        for key in [
            "image_size", "channels", "T", "beta_start", "beta_end",
            "base_ch", "ch_mults", "batch_size", "lr", "total_steps",
            "ema_decay", "log_every", "ckpt_every", "ckpt_dir", "device"
        ]
    }
    config.update({
        "training_seed": TRAIN_SEED,
        "torch_version": torch.__version__,
        "gpu": torch.cuda.get_device_name(0),
        "git_commit": subprocess.check_output(
            ["git", "rev-parse", "HEAD"], text=True
        ).strip(),
    })
    
    (train_dir / "training_config.json").write_text(
        json.dumps(config, indent=2, ensure_ascii=False)
    )
    
    ddpm_mnist.train(ExperimentCFG)

if RUN_TRAINING:
    train_new_model(TRAIN_SEED)
else:
    print('已跳過訓練')


## 5. 可選：三種裁切方式的評估

以原始儲存格為基礎，保留取樣、FID、計時方式；加入完成步數與結果設定檢查，整理版結果另存。這份 Notebook 每次執行一個取樣 seed，重跑其他 seeds 時只修改 SAMPLING_SEED。中斷的單一組會從頭計算，已完成組別可跳過。

所有方法最後都裁切再轉 PNG；no_clip 指不做中間裁切。預覽 seed=2026 與 FID 取樣 seed 不同。

In [ ]:
def run_ablation(checkpoint_file, sampling_seed, eval_steps):
    # 裁切消融：直接在目前 Colab 執行
    
    import gc
    import json
    import time
    import hashlib
    import tempfile
    import shutil
    from pathlib import Path
    
    import torch
    from torchvision.utils import save_image
    from cleanfid import fid
    from ddpm_mnist import CFG, UNet, Diffusion
    from fid_eval import prepare_real_images, REAL_DIR
    
    DEVICE = "cuda"
    STEPS = list(eval_steps)
    MODES = ["no_clip", "clip_original", "clip_recompute"]
    NUM_IMAGES = 10_000
    BATCH_SIZE = 100
    SEED = sampling_seed
    
    checkpoint_path = Path(checkpoint_file)
    checkpoint_hash = hashlib.sha256(checkpoint_path.read_bytes()).hexdigest()
    
    CFG.device = DEVICE
    diffusion = Diffusion(CFG)
    model = UNet(CFG).to(DEVICE)
    
    checkpoint = torch.load(
        checkpoint_path, map_location="cpu", weights_only=True
    )
    assert checkpoint["step"] == 30000, "需要完成 30000 步的權重"
    assert "ema" in checkpoint
    model.load_state_dict(checkpoint["ema"], strict=True)
    model.eval()
    del checkpoint
    gc.collect()
    
    # 獨立實驗資料夾；同設定重新執行時可以跳過已完成組別
    experiment = {
        "version": 2,
        "source_commit": SOURCE_COMMIT,
        "completed_steps": 30000,
        "checkpoint_sha256": checkpoint_hash,
        "steps": STEPS,
        "modes": MODES,
        "num_images": NUM_IMAGES,
        "batch_size": BATCH_SIZE,
        "seed": SEED,
        "T": CFG.T,
        "beta_start": CFG.beta_start,
        "beta_end": CFG.beta_end,
        "image_size": CFG.image_size,
        "channels": CFG.channels,
        "torch": torch.__version__,
        "gpu": torch.cuda.get_device_name(0),
    }
    experiment_id = hashlib.sha256(
        json.dumps(experiment, sort_keys=True).encode()
    ).hexdigest()[:12]
    
    output_dir = checkpoint_path.parent / f"clipping_ablation_{experiment_id}"
    output_dir.mkdir(parents=True, exist_ok=True)
    results_path = output_dir / "results.json"
    
    if results_path.exists():
        report = json.loads(results_path.read_text())
        assert report["config"] == experiment, "已存結果設定不一致"
    else:
        report = {"config": experiment, "results": {}}
    
    def save_report():
        temporary = results_path.with_suffix(".tmp")
        temporary.write_text(
            json.dumps(report, indent=2, ensure_ascii=False)
        )
        temporary.replace(results_path)
    
    save_report()
    
    
    @torch.inference_mode()
    def sample(initial_noise, num_steps, mode):
        """eta=0；除裁切方式外，其餘更新與原 sampler 相同。"""
        x = initial_noise.clone()
        times = torch.linspace(
            diffusion.T - 1, 0, num_steps, dtype=torch.long
        ).tolist()
    
        for i, t in enumerate(times):
            previous = times[i + 1] if i + 1 < len(times) else -1
            ab = diffusion.alpha_bars[t]
            ab_previous = (
                diffusion.alpha_bars[previous]
                if previous >= 0
                else x.new_tensor(1.0)
            )
    
            t_batch = torch.full(
                (x.shape[0],), t, device=DEVICE, dtype=torch.long
            )
            eps = model(x, t_batch)
            x0 = (x - (1 - ab).sqrt() * eps) / ab.sqrt()
    
            if mode != "no_clip":
                x0 = x0.clamp(-1, 1)
    
            if mode == "clip_recompute":
                eps = (x - ab.sqrt() * x0) / (1 - ab).sqrt()
    
            x = ab_previous.sqrt() * x0 + (1 - ab_previous).sqrt() * eps
    
        return x
    
    
    prepare_real_images()
    
    # 固定圖片對照用的起始噪聲
    preview_generator = torch.Generator(device=DEVICE).manual_seed(2026)
    preview_noise = torch.randn(
        16, CFG.channels, CFG.image_size, CFG.image_size,
        device=DEVICE, generator=preview_generator
    )
    
    for mode in MODES:
        for steps in STEPS:
            key = f"{mode}_{steps}"
    
            if key in report["results"]:
                print("已完成，跳過：", key)
                continue
    
            print(f"\n開始：{key}", flush=True)
    
            # 暖機，不納入時間
            sample(preview_noise, steps, mode)
            torch.cuda.synchronize()
    
            # 每組重設同一 seed，確保使用相同起始噪聲
            generator = torch.Generator(device=DEVICE).manual_seed(SEED)
            fake_dir = Path(tempfile.mkdtemp(prefix="ddim_ablation_"))
            generation_seconds = 0.0
            outside_pixels = 0
            total_pixels = 0
    
            try:
                for start in range(0, NUM_IMAGES, BATCH_SIZE):
                    n = min(BATCH_SIZE, NUM_IMAGES - start)
                    noise = torch.randn(
                        n, CFG.channels, CFG.image_size, CFG.image_size,
                        device=DEVICE, generator=generator
                    )
    
                    torch.cuda.synchronize()
                    begin = time.perf_counter()
                    images = sample(noise, steps, mode)
                    torch.cuda.synchronize()
                    generation_seconds += time.perf_counter() - begin
    
                    if not torch.isfinite(images).all().item():
                        raise RuntimeError(f"{key} 產生 NaN 或 Inf")
    
                    outside_pixels += (
                        (images < -1) | (images > 1)
                    ).sum().item()
                    total_pixels += images.numel()
    
                    # 所有方法都以同樣方式轉為合法像素供 FID 評估。
                    # no_clip 指的是不做「中間步驟」裁切。
                    images = ((images.clamp(-1, 1) + 1) / 2).cpu()
                    for j, img in enumerate(images):
                        save_image(img, fake_dir / f"{start+j:05d}.png")
    
                    if (start + n) % 2000 == 0:
                        print(f"  已生成 {start+n}/{NUM_IMAGES}", flush=True)
    
                score = float(fid.compute_fid(
                    str(REAL_DIR), str(fake_dir),
                    mode="clean", device=torch.device(DEVICE)
                ))
    
                preview = sample(preview_noise, steps, mode)
                save_image(
                    ((preview.clamp(-1, 1) + 1) / 2).cpu(),
                    output_dir / f"{key}.png",
                    nrow=4
                )
    
                report["results"][key] = {
                    "mode": mode,
                    "steps": steps,
                    "fid": score,
                    "generation_seconds": generation_seconds,
                    "images_per_second": NUM_IMAGES / generation_seconds,
                    "final_out_of_range_fraction": outside_pixels / total_pixels,
                }
                save_report()
                print(
                    f"完成：FID={score:.4f}, "
                    f"生成時間={generation_seconds:.1f} 秒",
                    flush=True
                )
            finally:
                shutil.rmtree(fake_dir, ignore_errors=True)
    
    print("\n全部完成！結果位置：", results_path)
    for key, result in report["results"].items():
        print(f"{key:24s} FID={result['fid']:.4f}")
    return results_path

if RUN_ABLATION:
    results_path = run_ablation(CHECKPOINT, SAMPLING_SEED, EVAL_STEPS)
    print('結果已保存：', results_path)
else:
    print('已跳過生成與 FID；已發表的結果見第 2 節')


## 6. 證據與限制

- 原始 Notebook 保留其輸出、錯誤與執行順序，作為過程紀錄；Notebook 的程式可能在輸出後被編輯，因此各輪數值以 results/ 中的 JSON 為準。
- 原始儲存格中可見 30,000 步與 EMA 檢查、最後一次 seed=123 的 FID 輸出；不將它描述為完整的六輪執行歷史。
- 本整理版僅完成結構與程式對照審查，尚未在 Colab 執行。重建圖表不等於重新驗證訓練或 FID。
- 兩個模型提供有限重現證據；取樣標準差不反映全部訓練變異、FID 偏差或資料集泛化。
- 推甄文件的作者貢獻、AI 協助範圍與專案期間須由申請人確認，不能由程式紀錄推定。
